In [1]:
import os
import cv2
import numpy as np
from sklearn.model_selection import train_test_split
 
# ── Patch BEFORE any keras import that touches BatchNormalization ──────────────
import tensorflow.keras.layers as _kl
_orig_bn_init = _kl.BatchNormalization.__init__
def _patched_bn_init(self, **kwargs):
    for k in ("renorm", "renorm_clipping", "renorm_momentum",
              "virtual_batch_size", "adjustment"):
        kwargs.pop(k, None)
    _orig_bn_init(self, **kwargs)
_kl.BatchNormalization.__init__ = _patched_bn_init
 
import tensorflow as tf
from tensorflow.keras import layers, models, Input
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

In [2]:
IMG_SIZE     = 64
WEIGHTS_PATH = "models/best_model.weights.h5"
DATA_DIR     = r"D:\DS\Projects\DL\facevision_ai\data\archive (7)\UTKFace"

In [3]:
def build_model(input_shape=(64, 64, 3)):
    """MobileNetV2 backbone + dual-head for gender (sigmoid) and age (linear)."""
    base = MobileNetV2(input_shape=input_shape, include_top=False, weights='imagenet')
    base.trainable = False
    inputs     = Input(shape=input_shape)
    x          = base(inputs, training=False)
    x          = layers.GlobalAveragePooling2D()(x)
    x          = layers.Dense(256, activation='relu')(x)
    x          = layers.Dropout(0.4)(x)
    gender_out = layers.Dense(1, activation='sigmoid', name='gender')(x)
    age_out    = layers.Dense(1, activation='linear',  name='age')(x)
    return models.Model(inputs, [gender_out, age_out])

In [5]:
def load_utk_data(data_dir, max_samples=None):
    """
    Load UTKFace images. Filename format: age_gender_race_timestamp.jpg
    gender: 0 = Male, 1 = Female
    age:    1 – 116 (normalised to 0–1 for training)
    """
    images, genders, ages = [], [], []
    files = [f for f in os.listdir(data_dir) if f.endswith('.jpg')]
    if max_samples:
        files = files[:max_samples]
    for fname in files:
        try:
            parts  = fname.split('_')
            age    = int(parts[0])
            gender = int(parts[1])
            if gender not in [0, 1] or not (1 <= age <= 116):
                continue
            img = cv2.imread(os.path.join(data_dir, fname))
            img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            images.append(img)
            genders.append(gender)
            ages.append(age)
        except Exception:
            continue
    X        = np.array(images,  dtype='float32') / 255.0
    y_gender = np.array(genders, dtype='float32')
    y_age    = np.array(ages,    dtype='float32') / 116.0
    return X, y_gender, y_age

In [6]:
def train(data_dir=DATA_DIR, epochs=5, batch_size=500,
          test_split=0.2, max_samples=None, extra_callbacks=None):
    """
    Full training pipeline. Returns (model, history).
    extra_callbacks: list of tf.keras.callbacks to append (e.g. Streamlit logger).
    """
    X, y_gender, y_age = load_utk_data(data_dir, max_samples)
 
    X_train, X_val, yg_train, yg_val, ya_train, ya_val = train_test_split(
        X, y_gender, y_age, test_size=test_split, random_state=42
    )
    print(f"Train: {X_train.shape} | Val: {X_val.shape}")
 
    model = build_model()
    model.compile(
        optimizer=Adam(1e-4),
        loss={'gender': 'binary_crossentropy', 'age': 'mae'},
        loss_weights={'gender': 1.0, 'age': 0.5},
        metrics={'gender': 'accuracy', 'age': 'mae'}
    )
 
    os.makedirs("models", exist_ok=True)
 
    callbacks = [
        EarlyStopping(patience=8, restore_best_weights=True),
        ReduceLROnPlateau(patience=4, factor=0.5, verbose=0),
    ]
    if extra_callbacks:
        callbacks.extend(extra_callbacks)
    history = model.fit(
        X_train, {'gender': yg_train, 'age': ya_train},
        validation_data=(X_val, {'gender': yg_val, 'age': ya_val}),
        epochs=epochs,
        batch_size=batch_size,
        callbacks=callbacks,
        verbose=1
    )
 
    model.save_weights(WEIGHTS_PATH)
    print(f"Weights saved → {WEIGHTS_PATH}")
    return model, history

In [7]:
def load_trained_model(weights_path=WEIGHTS_PATH):
    """Load model with saved weights. Returns None if weights not found."""
    if not os.path.exists(weights_path):
        return None
    m = build_model()
    m.load_weights(weights_path)
    return m
 
 
def predict_age_gender(face_bgr, model):
   
    face = cv2.resize(face_bgr, (IMG_SIZE, IMG_SIZE))
    face = cv2.cvtColor(face, cv2.COLOR_BGR2RGB) / 255.0
    face = np.expand_dims(face, axis=0)
    gender_pred, age_pred = model.predict(face, verbose=0)
    raw        = float(gender_pred[0][0])
    gender     =  "Female" if raw > 0.5 else "Male"
    confidence = raw if raw > 0.5 else 1 - raw
    age        = int(age_pred[0][0] * 116)
    return gender, confidence, age
def detect_faces(frame_bgr, yolo=None):
    """YOLO face detector with Haar cascade fallback."""
    if yolo is not None:
        results = yolo(frame_bgr, verbose=False)[0]
        boxes = [tuple(box[:4].astype(int)) for box in results.boxes.xyxy.cpu().numpy()]
        if boxes:
            return boxes
    gray    = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2GRAY)
    cascade = cv2.CascadeClassifier(
        cv2.data.haarcascades + "haarcascade_frontalface_default.xml")
    dets = cascade.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5, minSize=(40, 40))
    return [] if len(dets) == 0 else [(x, y, x+w, y+h) for x, y, w, h in dets]
 
 
def apply_aging(face_rgb, intensity=0.5):
    """Simple aging effect: Gaussian noise + greyscale blend."""
    noise    = np.random.normal(0, 25 * intensity, face_rgb.shape)
    aged     = np.clip(face_rgb.astype(float) + noise, 0, 255).astype(np.uint8)
    grey_3ch = cv2.cvtColor(cv2.cvtColor(aged, cv2.COLOR_RGB2GRAY), cv2.COLOR_GRAY2RGB)
    return cv2.addWeighted(aged, 1 - intensity * 0.4, grey_3ch, intensity * 0.4, 0)

In [ ]:
if __name__ == "__main__":
    train()

Train: (18966, 64, 64, 3) | Val: (4742, 64, 64, 3)


C:\Users\ALEENA THOMAS\AppData\Local\Temp\ipykernel_15812\3885412283.py:3: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  base = MobileNetV2(input_shape=input_shape, include_top=False, weights='imagenet')


Epoch 1/5
38/38 ━━━━━━━━━━━━━━━━━━━━ 59s 1s/step - age_loss: 0.8587 - age_mae: 0.8589 - gender_accuracy: 0.6098 - gender_loss: 0.7580 - loss: 1.1877 - val_age_loss: 0.3932 - val_age_mae: 0.3931 - val_gender_accuracy: 0.7296 - val_gender_loss: 0.5470 - val_loss: 0.7444 - learning_rate: 1.0000e-04
Epoch 2/5
38/38 ━━━━━━━━━━━━━━━━━━━━ 53s 1s/step - age_loss: 0.6012 - age_mae: 0.6014 - gender_accuracy: 0.7057 - gender_loss: 0.5847 - loss: 0.8855 - val_age_loss: 0.2963 - val_age_mae: 0.2961 - val_gender_accuracy: 0.7554 - val_gender_loss: 0.5101 - val_loss: 0.6595 - learning_rate: 1.0000e-04
Epoch 3/5
38/38 ━━━━━━━━━━━━━━━━━━━━ 57s 2s/step - age_loss: 0.4315 - age_mae: 0.4316 - gender_accuracy: 0.7312 - gender_loss: 0.5375 - loss: 0.7533 - val_age_loss: 0.2389 - val_age_mae: 0.2388 - val_gender_accuracy: 0.7735 - val_gender_loss: 0.4939 - val_loss: 0.6146 - learning_rate: 1.0000e-04
Epoch 4/5
38/38 ━━━━━━━━━━━━━━━━━━━━ 51s 1s/step - age_loss: 0.3287 - age_mae: 0.3288 - gender_accuracy: 0.75

In [ ]:
# ── Cell: Test on an image ─────────────────────────────────────────────────────
import cv2
import numpy as np

# Load your trained model (weights already saved from training above)
model = load_trained_model(r"D:\DS\Projects\DL\facevision_ai\models\best_model.weights.h5")

# Load any image from your UTKFace folder
img = cv2.imread(r"SaiPallavi.jpg")

# Detect face and predict
boxes = detect_faces(img)
if boxes:
    x1, y1, x2, y2 = boxes[0]
    face = img[y1:y2, x1:x2]
    gender, confidence, age = predict_age_gender(face, model)

    label = f"{gender}, {age}yrs ({int(confidence*100)}%)"
    cv2.rectangle(img, (x1, y1), (x2, y2), ((255, 0, 0)), 2)
    cv2.putText(img, label, (x1+5, y1+ 17),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0,0,255), 2)
    print(label)
else:
    print("No face detected")

cv2.imshow("FaceVision", img)
cv2.waitKey(0)
cv2.destroyAllWindows()

C:\Users\ALEENA THOMAS\AppData\Local\Temp\ipykernel_40672\3885412283.py:3: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  base = MobileNetV2(input_shape=input_shape, include_top=False, weights='imagenet')


Female, 27yrs (98%)


In [ ]:
# import cv2

# cap = cv2.VideoCapture(0) 
# print("Webcam started — press Q to quit")

# while True:
#     ret, frame = cap.read()
#     if not ret:
#         print("Cannot read webcam")
#         break

#     boxes = detect_faces(frame)
#     for (x1, y1, x2, y2) in boxes:
#         face = frame[y1:y2, x1:x2]
#         if face.size == 0:
#             continue
#         gender, confidence, age = predict_age_gender(face, model)
#         label = f"{gender}, {age}yrs ({int(confidence*100)}%)"
#         cv2.rectangle(frame, (x1, y1), (x2, y2), (128, 0, 255), 2)
#         cv2.putText(frame, label, (x1, y1 - 10),
#                     cv2.FONT_HERSHEY_SIMPLEX, 0.7, (128, 0, 255), 2)

#     cv2.imshow("FaceVision - Webcam  |  Q to quit", frame)
#     if cv2.waitKey(1) & 0xFF == ord('q'):
#         break

# cap.release()
# cv2.destroyAllWindows()

Webcam started — press Q to quit


In [13]:
import cv2
import numpy as np
from collections import deque

cap = cv2.VideoCapture(0)

age_buffer    = deque(maxlen=20)
gender_buffer = deque(maxlen=20)

while True:
    ret, frame = cap.read()
    if not ret:
        break

    boxes = detect_faces(frame)
    for (x1, y1, x2, y2) in boxes:
        face = frame[y1:y2, x1:x2]
        if face.size == 0:
            continue
        gender, confidence, age = predict_age_gender(face, model)

        age_buffer.append(age)
        gender_buffer.append(1 if gender == "Female" else 0)

        smooth_age    = int(np.mean(age_buffer))
        smooth_gender = "Female" if np.mean(gender_buffer) > 0.5 else "Male"

        print(f"Gender: {smooth_gender} | Age: {smooth_age} yrs", end="\r")  # ← added

        label = f"{smooth_gender}, {smooth_age}yrs ({int(confidence*100)}%)"
        cv2.rectangle(frame, (x1, y1), (x2, y2), (128, 0, 255), 2)
        cv2.putText(frame, label, (x1, y1 - 10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (128, 0, 255), 2)

    cv2.imshow("FaceVision - Webcam  |  Q to quit", frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()